# AI Presales Lab：免费 Colab GPU 的 QLoRA 实验

这个 notebook 将完成：

1. 检查 Colab GPU、CUDA 和显存；
2. 从 GitHub 拉取项目并安装不覆盖 Colab 自带 PyTorch 的训练依赖；
3. 重建并校验 case-level split 的微调数据；
4. 执行 QLoRA dry-run，再运行 Qwen2.5-0.5B-Instruct 的 TRL + PEFT 训练；
5. 在同一份 held-out test split 上比较 base model 和 adapter；
6. 保存运行环境、配置、manifest、metrics、评估报告和 adapter。

Colab 的 GPU 类型、可用时间和资源配额会动态变化。这个实验只能证明本次 runtime 的结果，不能直接当成生产 SLA。不要上传客户资料、API Key、访问 Token 或未脱敏的 notebook 输出。

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone

# 必须由你填写：建议使用公开 GitHub 仓库，方便面试官复现。
PROJECT_REPO_URL = 'PASTE_YOUR_PUBLIC_GITHUB_REPO_URL_HERE'

# True：训练/评估结束后把报告和 adapter 复制到 Google Drive；
# False：最后会生成 zip 并尝试下载，runtime 断开后 /content 文件会消失。
USE_DRIVE = True
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/ai-presales-lab-results')
# 首次运行保持 False；如果显存 OOM，改成 True 使用 1 batch / 1024 tokens 的降级配置。
LOW_MEMORY = False

# 模型公开可下载，不需要 Hugging Face Token。把缓存放在本地磁盘，避免 Drive I/O 拖慢训练。
os.environ['HF_HOME'] = '/content/huggingface-cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

if not PROJECT_REPO_URL.startswith('https://') or 'PASTE_' in PROJECT_REPO_URL:
    raise ValueError('请先把 PROJECT_REPO_URL 改成你的 GitHub 仓库地址，再运行本单元格。')

WORKDIR = Path('/content/ai-presales-qlora')
PROJECT_DIR = WORKDIR / 'ai-presales-lab'
REPORT_DIR = PROJECT_DIR / 'data' / 'results' / 'colab' / 'qlora'
WORKDIR.mkdir(parents=True, exist_ok=True)
print('workdir:', WORKDIR)
print('project:', PROJECT_DIR)

In [ ]:
# 可选但推荐：Colab runtime 是临时的，挂载 Drive 便于保存结果。
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print('Drive output:', DRIVE_RESULTS_DIR)
else:
    print('Drive disabled; final zip must be downloaded before disconnecting the runtime.')

In [ ]:
# 拉取代码。不要在 Colab 里把 Token 写进 URL。
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print(subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# GPU preflight：免费 Colab 通常会分配 NVIDIA GPU，但型号不保证。
gpu_probe = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(gpu_probe.stdout)
if gpu_probe.returncode != 0:
    print(gpu_probe.stderr)

import torch
assert torch.cuda.is_available(), '没有检测到 CUDA GPU：请在 Runtime > Change runtime type 中选择 GPU。'
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
print({
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': gpu_name,
    'gpu_memory_gb': gpu_memory_gb,
    'bf16_supported': bool(torch.cuda.is_bf16_supported()),
    'compute_dtype': 'float16 for Tesla T4; the project config pins this explicitly',
})
if gpu_memory_gb < 12:
    print('提示：显存低于 12GB 时先保持默认小模型和 batch size；发生 OOM 再按 Runbook 降参。')

In [ ]:
# 安装项目和 Colab 专用训练依赖。finetune-colab 不主动覆盖 Colab 自带 torch。
REPORT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[finetune-colab]'], check=True)
print('training dependencies installed')
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=(REPORT_DIR / 'pip-freeze.txt').open('w'))

In [ ]:
# 保存 runtime 元数据，后续简历只使用这次真实运行生成的数字。
REPORT_DIR.mkdir(parents=True, exist_ok=True)
runtime = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': gpu_name,
    'gpu_memory_gb': gpu_memory_gb,
    'bf16_supported': bool(torch.cuda.is_bf16_supported()),
}
(REPORT_DIR / 'runtime.json').write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(runtime, ensure_ascii=False, indent=2))

In [ ]:
# 生成本次 runtime 使用的配置。训练输出直接写入 Drive，runtime 中断后仍可找到 checkpoint。
config_name = 'trl_qlora_colab_lowmem.json' if LOW_MEMORY else 'trl_qlora.json'
base_config_path = PROJECT_DIR / 'configs' / 'finetune' / config_name
runtime_config_path = WORKDIR / 'trl_qlora_runtime.json'
training_config = json.loads(base_config_path.read_text(encoding='utf-8'))
if USE_DRIVE:
    train_output_name = 'active-training-lowmem' if LOW_MEMORY else 'active-training'
    train_output_dir = DRIVE_RESULTS_DIR / train_output_name
else:
    train_output_dir = PROJECT_DIR / '.runtime' / 'models' / 'qwen2.5-0.5b-presales-lora'
training_config['output_dir'] = str(train_output_dir)
runtime_config_path.write_text(json.dumps(training_config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('training config:', runtime_config_path)
print('checkpoint/adapter output:', train_output_dir)

In [ ]:
# 重建并校验训练数据；生成过程是确定性的，不会读取客户数据。
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
subprocess.run([sys.executable, 'scripts/build_finetune_dataset.py'], check=True, env=env)
subprocess.run([sys.executable, 'scripts/check_finetune_dataset.py'], check=True, env=env)
print((PROJECT_DIR / 'data' / 'finetuning' / 'manifest.json').read_text(encoding='utf-8'))

In [ ]:
# 先 dry-run，再训练。dry-run 失败时不要继续消耗 GPU。
subprocess.run([sys.executable, 'scripts/train_qlora.py', '--config', str(runtime_config_path), '--dry-run'], check=True, env=env)

In [ ]:
# 正式训练：默认 Qwen2.5-0.5B-Instruct、NF4、LoRA r=16、3 epochs。
# 若 runtime 中断，重新运行到本单元格前，先把 RESUME_CHECKPOINT 改成存在的 checkpoint-* 目录。
RESUME_CHECKPOINT = None
if USE_DRIVE:
    print('可用 checkpoint:', sorted(str(path) for path in train_output_dir.glob('checkpoint-*')))
train_command = [
    sys.executable, 'scripts/train_qlora.py',
    '--config', str(runtime_config_path),
]
if RESUME_CHECKPOINT:
    train_command += ['--resume-from-checkpoint', str(RESUME_CHECKPOINT)]
subprocess.run(train_command, check=True, env=env)
adapter_dir = train_output_dir
assert (adapter_dir / 'adapter_config.json').exists(), f'adapter not found: {adapter_dir}'
print('adapter:', adapter_dir)

In [ ]:
# 在完全相同的 held-out test split 上评估 base 与 adapter。
test_file = 'data/finetuning/test.jsonl'
base_report = REPORT_DIR / 'base_eval.json'
adapter_report = REPORT_DIR / 'adapter_eval.json'
subprocess.run([
    sys.executable, 'scripts/evaluate_finetuned_model.py',
    '--split', test_file, '--max-new-tokens', '1024',
    '--output', str(base_report),
], check=True, env=env)
subprocess.run([
    sys.executable, 'scripts/evaluate_finetuned_model.py',
    '--split', test_file, '--max-new-tokens', '1024',
    '--adapter', str(adapter_dir),
    '--output', str(adapter_report),
], check=True, env=env)

def metrics(path):
    return json.loads(path.read_text(encoding='utf-8'))['metrics']
print('base:', json.dumps(metrics(base_report), ensure_ascii=False, indent=2))
print('adapter:', json.dumps(metrics(adapter_report), ensure_ascii=False, indent=2))

In [ ]:
# 保存到 Drive；如果不使用 Drive，则生成 zip 并下载。
run_name = datetime.now().strftime('%Y%m%d-%H%M%S')
bundle_dir = WORKDIR / f'qlora-result-{run_name}'
bundle_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPORT_DIR, bundle_dir / 'reports', dirs_exist_ok=True)
shutil.copytree(adapter_dir, bundle_dir / 'adapter', dirs_exist_ok=True)
shutil.copy(runtime_config_path, bundle_dir / 'trl_qlora.json')
shutil.copy(PROJECT_DIR / 'data' / 'finetuning' / 'manifest.json', bundle_dir / 'manifest.json')
archive = shutil.make_archive(str(bundle_dir), 'zip', root_dir=bundle_dir)
if USE_DRIVE:
    target = DRIVE_RESULTS_DIR / run_name
    shutil.copytree(bundle_dir, target, dirs_exist_ok=True)
    print('saved to Drive:', target)
else:
    from google.colab import files
    files.download(archive)
    print('downloaded:', archive)

## 实验结束后的记录

你需要从 `reports/runtime.json`、`reports/base_eval.json`、`reports/adapter_eval.json` 和 adapter 目录中整理：GPU 型号、显存、训练时间、train/eval loss、JSON parse rate、schema pass rate、policy pass rate、保守无证据数量和 OOM/中断情况。

只有当 base 与 adapter 使用同一个 test split、相同生成参数，并且结果已保存为 JSON 后，才把效果差异写进简历。训练没有跑完、输出无法解析或只在训练集上变好，都不能写成模型效果提升。